# 01 — Data validation and experiment integrity

This notebook must be completed before treatment effects are interpreted. It validates the source file, treatment allocation, baseline balance and outcome consistency without silently changing the raw data.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_validation import (
    load_data,
    numeric_balance_table,
    validate_data,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

## Download the source data

Run the repository download script from a terminal before continuing:

```bash
python scripts/download_data.py
```

In [ ]:
data_path = ROOT / 'data' / 'raw' / 'hillstrom.csv'
df = load_data(data_path)
df.shape, df.head()

## Source-data checks

The report records issues for review. It does not automatically delete or repair rows.

In [ ]:
validation = validate_data(df)
validation_table = (
    pd.Series(validation.to_dict(), name='value')
    .rename_axis('check')
    .reset_index()
)
validation_table

## Treatment allocation

The chi-square sample-ratio test is a diagnostic. A small p-value would trigger investigation rather than an automatic conclusion about the campaign.

In [ ]:
allocation = (
    df['segment']
    .value_counts()
    .rename('customers')
    .to_frame()
)
allocation['allocation_pct'] = allocation['customers'] / len(df)
allocation

## Numeric and binary baseline balance

Standardised mean differences are used alongside descriptive distributions. With a large sample, p-values can flag differences too small to matter operationally.

In [ ]:
balance = numeric_balance_table(df)
balance['absolute_smd'] = balance['standardised_mean_difference'].abs()
balance.sort_values('absolute_smd', ascending=False)

## Categorical baseline distributions

In [ ]:
categorical_fields = ['history_segment', 'zip_code', 'channel']
categorical_balance = {}
for field in categorical_fields:
    categorical_balance[field] = pd.crosstab(
        df[field],
        df['segment'],
        normalize='columns',
    )
categorical_balance

## Export validated tables

Only validated summaries are exported for downstream reporting.

In [ ]:
output_dir = ROOT / 'reports' / 'tables'
output_dir.mkdir(parents=True, exist_ok=True)
validation_table.to_csv(output_dir / 'validation_report.csv', index=False)
allocation.reset_index(names='segment').to_csv(
    output_dir / 'treatment_allocation.csv', index=False
)
balance.to_csv(output_dir / 'numeric_balance.csv', index=False)
print(f'Exported validation tables to {output_dir}')